In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

# 1. Load the final dataset
df = pd.read_csv("final_sih_master_training_dataset.csv")

# 2. Define the target variable (continuous stress percentage)
y = df["overall_stress_score"]

# 3. Drop only the actual non-predictive/target columns to isolate features (X)
X = df.drop(columns=["timestamp", "target_diagnosis", "overall_stress_score"])

# 4. Split the data (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Initialize and train the Random Forest model
print("Training Random Forest model...")
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# 6. Evaluate the model
predictions = rf_model.predict(X_test)
print(f"\nMean Absolute Error (MAE): {mean_absolute_error(y_test, predictions):.2f}")
print(f"R-Squared (R2) Score: {r2_score(y_test, predictions):.4f}")

# 7. Extract Feature Importances
importance_df = pd.DataFrame({
    "Feature": X.columns, 
    "Importance": rf_model.feature_importances_
}).sort_values(by="Importance", ascending=False).reset_index(drop=True)

print("\n--- TOP 5 STRESS DRIVERS ---")
print(importance_df.head(5))

# 8. Export the trained model for deployment
joblib.dump(rf_model, "sih_stress_rf_model.pkl")

Training Random Forest model...

Mean Absolute Error (MAE): 2.33
R-Squared (R2) Score: 0.9583

--- TOP 5 STRESS DRIVERS ---
               Feature  Importance
0             rmssd_ms    0.645509
1          relax_hours    0.149069
2    behavior_subscore    0.068062
3       voice_subscore    0.025760
4  baseline_blink_rate    0.024670


['sih_stress_rf_model.pkl']

In [3]:

# 1. Load the trained model from the file
loaded_model = joblib.load("sih_stress_rf_model.pkl")
print("Model loaded successfully!")

# 2. Create a dummy input matching the EXACT 16 features the model was trained on
mock_data = {
    "hr_bpm": [88.5],
    "relax_hours": [5.2],
    "bp_systolic": [135.0],
    "bp_diastolic": [89.0],
    "body_weight_kg": [78.4],
    "duty_hours": [14.5],
    "rmssd_ms": [22.4],          
    "baseline_blink_rate": [28.0],
    "pitch_mean_hz": [145.2],
    "brow_ratio": [0.15],
    "head_jitter": [110.5],
    "pitch_std_hz": [25.4],
    "speech_blink_rate": [35.2],
    "hrv_subscore": [40.5],
    "voice_subscore": [65.2],
    "behavior_subscore": [55.8]  
}

# 3. Convert the dictionary to a Pandas DataFrame
input_df = pd.DataFrame(mock_data)

# 4. Run the prediction
predicted_stress = loaded_model.predict(input_df)

print(f"\n--- PREDICTION RESULT ---")
print(f"Predicted Overall Stress Score: {predicted_stress[0]:.1f}%")

Model loaded successfully!

--- PREDICTION RESULT ---
Predicted Overall Stress Score: 81.3%
